# Inference / Checkpoint Evaluation (Colab / Local)

Loads a checkpoint trained by `03_train_colab.ipynb` and evaluates it on the test split
two ways: the standard **single-window** sample each clip got during training, and
**multi-clip** (`MultiClipWorkoutDataset` + `evaluate_multi_clip`, in `training_utils.py`) -
averaging predictions over `NUM_CLIPS` windows spanning the whole clip. Compares the two so
we can see whether multi-clip actually helps on this dataset before relying on it.

Does not retrain anything - only needs `artifacts/<EXP_NAME>/checkpoints/*.ckpt` to already
exist (from a previous run of `03_train_colab.ipynb` with a matching `exp_name`), locally
or on Colab.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Works around a known Windows conda/pip OpenMP DLL conflict (harmless elsewhere).
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

GIT_URL = 'https://github.com/hagairavid18/beilinson.git'
GIT_BRANCH = 'main'
COLAB_DIR = '/content/beilinson'

try:
    import google.colab  # noqa: F401
    on_colab = True
except ImportError:
    on_colab = False

if on_colab:
    PROJECT_ROOT = Path(COLAB_DIR)
    if not PROJECT_ROOT.exists():
        subprocess.check_call(['git', 'clone', '--branch', GIT_BRANCH, GIT_URL, str(PROJECT_ROOT)])
else:
    # Not on Colab (e.g. a local kernel) - use the repo checkout we're already in.
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

if on_colab:
    # Self-healing pull (recovers from diverged history, e.g. a commit made directly in
    # a previous Colab session) - see training_utils.sync_repo.
    from training_utils import sync_repo
    sync_repo(PROJECT_ROOT, GIT_BRANCH)

print('Project root:', PROJECT_ROOT)
print('Has data already:', any((PROJECT_ROOT / 'data').glob('*/*')))

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

In [ ]:
import json

import yaml
import pandas as pd
import lightning.pytorch as pl
from torch.utils.data import DataLoader

from dataset import MultiClipWorkoutDataset, WorkoutSequenceDataset
from model import SequenceClassifier
from pytorch_lightning import WorkoutLightningModule
from training_utils import (
    classification_metrics,
    ensure_artifacts,
    ensure_dataset,
    ensure_image_cache,
    evaluate_multi_clip,
    plot_confusion_matrix,
)

## Config

`CONFIG_PATH` picks which yaml file under `configs/` to read defaults from - only matters
as a fallback for checkpoints without an accompanying `training_summary.json` (see below).
`EXP_NAME` comes from the config's own `exp_name` field (not the filename) and must match
whatever `exp_name` `03_train_colab.ipynb` used for that run.

In [ ]:
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'base.yaml'

with open(CONFIG_PATH, 'r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

# Which experiment's output folder (artifacts/<EXP_NAME>/) to evaluate a checkpoint from -
# matches whatever 03_train_colab.ipynb's CONFIG['exp_name'] was for that run.
EXP_NAME = CONFIG['exp_name']
exp_dir = PROJECT_ROOT / 'artifacts' / EXP_NAME

# How many windows to average per clip for the multi-clip evaluation below.
NUM_CLIPS = 5

print('Using config:', CONFIG_PATH)
print('Experiment dir:', exp_dir)

## Dataset

In [ ]:
class_names = ensure_dataset(PROJECT_ROOT)
print(f'{len(class_names)} classes:', class_names)

## Pick a checkpoint

Defaults to the `best_model_path` recorded by the last `03_train_colab.ipynb` run for
this experiment (`exp_dir/training_summary.json`); falls back to the most recently
modified `.ckpt` under `exp_dir/checkpoints/`, then to the legacy flat
`artifacts/checkpoints/` (pre-per-experiment-folders) for older runs. Set
`CHECKPOINT_PATH` yourself to evaluate a different one.

Also pulls `model_config` (which backbone etc. that run actually used) from
`training_summary.json` if present, since `configs/base.yaml`'s defaults may have changed
since that checkpoint was trained. Falls back to the live `CONFIG['model']` above for
checkpoints saved before this existed (e.g. downloaded/committed by hand).

In [ ]:
summary_path = exp_dir / 'training_summary.json'
checkpoint_dir = exp_dir / 'checkpoints'

CHECKPOINT_PATH = None
MODEL_CONFIG = CONFIG['model']

if summary_path.exists():
    with open(summary_path, 'r', encoding='utf-8') as handle:
        train_summary = json.load(handle)
    best_model_path = train_summary.get('best_model_path', '')
    if best_model_path and Path(best_model_path).exists():
        CHECKPOINT_PATH = Path(best_model_path)
    if train_summary.get('model_config'):
        MODEL_CONFIG = train_summary['model_config']

if CHECKPOINT_PATH is None:
    checkpoints = sorted(checkpoint_dir.glob('*.ckpt'), key=lambda p: p.stat().st_mtime)
    if not checkpoints:
        # Fall back to the old flat layout, from before per-experiment folders existed.
        legacy_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints'
        checkpoints = sorted(legacy_dir.glob('*.ckpt'), key=lambda p: p.stat().st_mtime)
    if not checkpoints:
        raise FileNotFoundError(
            f'No checkpoints found under {checkpoint_dir} or the legacy artifacts/checkpoints/ '
            '- run 03_train_colab.ipynb first.'
        )
    CHECKPOINT_PATH = checkpoints[-1]

print('Using checkpoint:', CHECKPOINT_PATH)
print('Using model config:', MODEL_CONFIG)

## Load model from checkpoint

`WorkoutLightningModule` ignores `model` in `save_hyperparameters`, so the architecture has
to be rebuilt (from `MODEL_CONFIG` above - the run's actual config when available) and
passed in explicitly - only `lr`/`weight_decay` are restored automatically from the checkpoint.

In [ ]:
artifacts = ensure_artifacts(CONFIG, PROJECT_ROOT)
label_map = pd.read_csv(artifacts['label_map'])
num_classes = int(label_map['label_id'].nunique())

model = SequenceClassifier(
    num_classes=num_classes,
    in_channels=MODEL_CONFIG['in_channels'],
    hidden_dims=tuple(MODEL_CONFIG['hidden_dims']),
    embedding_dim=MODEL_CONFIG['embedding_dim'],
    dropout=MODEL_CONFIG['dropout'],
    temporal_pooling=MODEL_CONFIG['temporal_pooling'],
    backbone=MODEL_CONFIG.get('backbone', 'custom'),
    freeze_backbone=MODEL_CONFIG.get('freeze_backbone', True),
    classifier_hidden_dim=MODEL_CONFIG.get('classifier_hidden_dim'),
    pretrained_backbone=False,  # loading a checkpoint next - no need to also download ImageNet weights first
)
lit_module = WorkoutLightningModule.load_from_checkpoint(str(CHECKPOINT_PATH), model=model)
lit_module.eval()

## Evaluate: single window (standard)

Same sampling every test clip got during training - one `sequence_len`-frame window,
evenly spaced across the whole clip.

In [ ]:
data_cfg = CONFIG['data']
cached_data_root = ensure_image_cache(PROJECT_ROOT, data_cfg['image_size'])

test_dataset = WorkoutSequenceDataset(
    artifacts['sequence_manifest'], cached_data_root, split='test', image_size=data_cfg['image_size'],
)
test_loader = DataLoader(test_dataset, batch_size=data_cfg['batch_size'], shuffle=False)

trainer = pl.Trainer(logger=False, enable_checkpointing=False, enable_progress_bar=True)
single_window_results = trainer.test(lit_module, dataloaders=test_loader, verbose=True)

## Evaluate: multi-clip

`NUM_CLIPS` windows per test clip (set above), predictions averaged per clip.

In [ ]:
multi_clip_dataset = MultiClipWorkoutDataset(
    frame_manifest_path=artifacts['frame_manifest'],
    label_map_path=artifacts['label_map'],
    data_root=cached_data_root,
    split='test',
    sequence_len=data_cfg['sequence_len'],
    num_clips=NUM_CLIPS,
    image_size=data_cfg['image_size'],
)
multi_clip_accuracy, multi_clip_results = evaluate_multi_clip(
    lit_module, multi_clip_dataset, batch_size=max(1, data_cfg['batch_size'] // NUM_CLIPS),
)
print(f'Multi-clip (num_clips={NUM_CLIPS}) test accuracy: {multi_clip_accuracy:.4f}')

## Compare

In [ ]:
single_window_accuracy = single_window_results[0]['test_acc']
delta = multi_clip_accuracy - single_window_accuracy

pd.DataFrame(
    [
        {'strategy': 'single window', 'num_clips': 1, 'test_acc': single_window_accuracy},
        {'strategy': 'multi-clip', 'num_clips': NUM_CLIPS, 'test_acc': multi_clip_accuracy},
    ]
)

## Confusion matrix & per-class metrics (multi-clip)

In [ ]:
class_names_ordered = label_map.sort_values('label_id')['class'].tolist()

confusion_df, report_df = classification_metrics(multi_clip_results, class_names_ordered)
plot_confusion_matrix(confusion_df, title=f'Test set confusion matrix (multi-clip, num_clips={NUM_CLIPS})')
report_df